# 📏 Python Intervals — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Intervals are like calendar bookings on a timeline.
> Some overlap, some don't, some are back-to-back.
> You sort them by when they start (or end) to decide which to keep, merge, or remove.
> Greedy works here: the earliest-ending booking leaves the most room for future ones.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [What Are Intervals? The Visual Model](#1) |
| 2 | [Setup / Initialization](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Merge Intervals — LC 56](#5) |
| 6 | [Pattern 2: Insert Interval — LC 57](#6) |
| 7 | [Pattern 3: Non-Overlapping Intervals — LC 435](#7) |
| 8 | [Pattern 4: Minimum Arrows — LC 452](#8) |
| 9 | [Pattern 5: Meeting Rooms — Sweep Line](#9) |
| 10 | [The Intervals Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>

## 1. What Are Intervals? The Visual Model

```
TIMELINE — intervals as bar segments

  0   1   2   3   4   5   6   7   8   9  10
  |   |   |   |   |   |   |   |   |   |   |
  [=======]               A=[1,3]
          [=======]       B=[3,5]
              [=======]   C=[4,6]
                          [=======]   D=[7,9]

  A and B share endpoint 3: TOUCHING (overlapping by standard def)
  B and C share [4,5]:      OVERLAPPING
  C and D: gap at 6-7       NO OVERLAP

  MERGED:  [1,6] and [7,9]

LC 57 INSERT — three zones

  existing = [[1,3],[6,9]]   new = [2,5]

  Zone 1 — BEFORE new (end < new.start):
    [1,3]: end=3 < 2? No. Actually 3 >= 2 → overlaps

  Zone 2 — OVERLAPPING (start <= new.end AND end >= new.start):
    [1,3] overlaps [2,5] → merge → [1,5]
    [6,9]: start=6 > 5 → stop

  Zone 3 — AFTER new (start > new.end):
    [6,9] appended as-is

  result = [[1,5],[6,9]]

SWEEP LINE — meeting rooms

  meetings: [0,30],[5,10],[15,20]
  events: +1 at 0, +1 at 5, -1 at 10, +1 at 15, -1 at 20, -1 at 30
  sort events by time → sweep → track peak concurrent count
```

<a id='2'></a>

## 2. Setup / Initialization

In [ ]:
from typing import List

# Sort intervals by start time
intervals = [[1,3],[6,9],[2,5],[8,10]]
sorted_by_start = sorted(intervals, key=lambda x: x[0])
print(f"sorted by start: {sorted_by_start}")

# Sort by end time (greedy interval scheduling)
sorted_by_end = sorted(intervals, key=lambda x: x[1])
print(f"sorted by end:   {sorted_by_end}")

# Overlap check — two intervals [a,b] and [c,d] overlap when a<=d and c<=b
def overlaps(a, b):
    return a[0] <= b[1] and b[0] <= a[1]   # neither is entirely before the other

print(f"[1,3] and [2,5] overlap: {overlaps([1,3],[2,5])}")  # True
print(f"[1,3] and [4,6] overlap: {overlaps([1,3],[4,6])}")  # False
print(f"[1,3] and [3,5] overlap: {overlaps([1,3],[3,5])}")  # True (touching)

# Merge two overlapping intervals
def merge_two(a, b):
    return [min(a[0], b[0]), max(a[1], b[1])]

print(f"merge [1,3] [2,5]: {merge_two([1,3],[2,5])}")
print("Setup demonstrated.")

<a id='3'></a>

## 3. The Core API — All Operations

```
OPERATION                                COMPLEXITY   WHAT IT DOES
─────────────────────────────────────────────────────────────────────
sorted(intervals, key=lambda x: x[0])   O(n log n)   sort by start
sorted(intervals, key=lambda x: x[1])   O(n log n)   sort by end
a[0] <= b[1] and b[0] <= a[1]           O(1)         overlap check
new_end = max(merged_end, a[1])          O(1)         extend merged interval
sweep line: +1 at start, -1 at end      O(n log n)   sort events + scan

OVERLAP RULE VARIANTS:
  strictly overlapping:    a[0] <  b[1] and b[0] <  a[1]
  touching counts as overlap: a[0] <= b[1] and b[0] <= a[1]  (LC 56)
  LC 435 (arrows inclusive): start <= arrow (touching = hit)

THINGS YOU DO NOT DO:
❌  Check overlap without sorting first — O(n²) comparisons
❌  Forget to handle the last merged interval (append after loop)
❌  Confuse 'sort by start' vs 'sort by end' — they serve different problems
❌  In LC 435/452 boundary case: touching intervals DO interact
❌  Advance mid after seeing a 2 in Dutch flag (wrong interval analogy)
```

In [ ]:
# Live demo: merge step by step
intervals = [[1,3],[2,6],[8,10],[15,18]]
intervals.sort(key=lambda x: x[0])
merged = [intervals[0]]

print("Merge trace:")
for curr in intervals[1:]:
    last = merged[-1]
    if curr[0] <= last[1]:             # current starts before last ends
        last[1] = max(last[1], curr[1])  # extend last to cover current
        print(f"  merged {curr} into {last}")
    else:
        merged.append(curr)            # gap — start a new interval
        print(f"  gap before {curr} — appended")

print(f"result: {merged}")
print()

# Sweep line demo
meetings = [[0,30],[5,10],[15,20]]
events = []
for s, e in meetings:
    events.append((s, 1))    # start event
    events.append((e, -1))   # end event
events.sort(key=lambda x: (x[0], x[1]))  # sort by time, ends before starts at tie

rooms = 0
peak = 0
for time, delta in events:
    rooms += delta
    peak = max(peak, rooms)

print(f"meetings={meetings}")
print(f"peak concurrent rooms: {peak}")
print("Core API demo done.")

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                      SORT BY   APPROACH
────────────────────────────────────────────────────────────────────
"merge overlapping intervals"              start     scan + extend last
"insert new interval into sorted list"     (already sorted) three-phase
"minimum removals to make non-overlapping" end       greedy keep earliest-end
"minimum arrows to burst all balloons"     end       greedy fire at first end
"min rooms for all meetings"               —         sweep line +1/-1 events
"can attend all meetings"                  start     check consecutive overlaps
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Merge Intervals — LC 56

---

```
PROBLEM:
  Merge all overlapping intervals and return the non-overlapping result.

TRICK:
  Sort by start. Walk through: if current start <= last merged end, extend.
  Otherwise append as new interval.

SLOW MOTION TRACE on [[1,3],[2,6],[8,10],[15,18]]:
  sorted: [[1,3],[2,6],[8,10],[15,18]]
  merged=[[1,3]]

  curr=[2,6]: 2<=3 → extend → merged=[[1,6]]
  curr=[8,10]: 8>6 → gap → merged=[[1,6],[8,10]]
  curr=[15,18]: 15>10 → gap → merged=[[1,6],[8,10],[15,18]]

  result=[[1,6],[8,10],[15,18]]

KEY INSIGHT:
  After sorting by start, you only need to compare with the LAST merged
  interval — earlier ones can never overlap with the current.

TIME:  O(n log n) — sort dominates
SPACE: O(n)       — output list
```

In [ ]:
def merge(intervals: List[List[int]]) -> List[List[int]]:
    """
    LC 56 — Merge Intervals
    Approach: sort by start, extend last merged interval on overlap.
    Args:
        intervals (List[List[int]]): list of [start, end] intervals.
    Returns:
        List[List[int]]: merged non-overlapping intervals.
    Time:  O(n log n) — sort dominates; merge pass is O(n)
    Space: O(n)       — output array
    """
    intervals.sort(key=lambda x: x[0])   # sort by start time
    merged = [intervals[0]]

    for curr in intervals[1:]:
        last = merged[-1]
        if curr[0] <= last[1]:            # overlap: current starts inside last
            last[1] = max(last[1], curr[1])  # extend last to cover current
        else:
            merged.append(curr)           # gap: start fresh interval

    return merged

# Slow motion on [[1,3],[2,6],[8,10],[15,18]]:
# [2,6]: 2<=3 → extend last to [1,6]
# [8,10]: 8>6 → append → [[1,6],[8,10]]
# [15,18]: 15>10 → append → [[1,6],[8,10],[15,18]]

def test_harness(fn):
    import copy
    tests = [
        ([[1,3],[2,6],[8,10],[15,18]], [[1,6],[8,10],[15,18]]),
        ([[1,4],[4,5]],               [[1,5]]),
        ([[1,4],[2,3]],               [[1,4]]),
        ([[1,2]],                     [[1,2]]),
        ([[1,4],[0,4]],               [[0,4]]),
        ([[1,4],[0,0]],               [[0,0],[1,4]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(copy.deepcopy(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(merge)
print("merge defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Insert Interval — LC 57

---

```
PROBLEM:
  Insert a new interval into a sorted non-overlapping list. Merge if needed.

TRICK:
  Three phases — no sort needed (list already sorted):
  1. BEFORE: intervals entirely before new (end < new.start) → copy as-is
  2. OVERLAP: intervals overlapping new → merge into new
  3. AFTER: intervals entirely after new (start > new.end) → copy as-is

SLOW MOTION TRACE on intervals=[[1,2],[3,5],[6,7],[8,10],[12,16]] new=[4,8]:
  Phase 1 — before new=[4,8]:
    [1,2]: end=2 < 4 → copy → result=[[1,2]]
    [3,5]: end=5 >= 4 → stop

  Phase 2 — overlapping:
    [3,5]: 3<=8 and 5>=4 → merge → new=[3,8]
    [6,7]: 6<=8 and 7>=4 → merge → new=[3,8]
    [8,10]: 8<=8 and 10>=4 → merge → new=[3,10]
    [12,16]: 12>8 → stop
  append merged new=[3,10] → result=[[1,2],[3,10]]

  Phase 3 — after:
    [12,16]: append → result=[[1,2],[3,10],[12,16]]

KEY INSIGHT:
  No sorting needed — the list is already ordered. Three clean phases.
  Overlap condition: existing.start <= new.end AND existing.end >= new.start.

TIME:  O(n) — one pass through intervals
SPACE: O(n) — output array
```

In [ ]:
def insert(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    """
    LC 57 — Insert Interval
    Approach: three-phase scan (before, overlap, after). No sorting needed.
    Args:
        intervals (List[List[int]]): sorted non-overlapping intervals.
        new (List[int]): new interval to insert.
    Returns:
        List[List[int]]: merged interval list after insertion.
    Time:  O(n) — single pass, no sort needed
    Space: O(n) — output array
    """
    result = []
    i = 0
    n = len(intervals)

    # Phase 1: intervals entirely BEFORE new (they end before new starts)
    while i < n and intervals[i][1] < new[0]:
        result.append(intervals[i])
        i += 1

    # Phase 2: overlapping intervals — merge all into new
    while i < n and intervals[i][0] <= new[1]:
        new[0] = min(new[0], intervals[i][0])  # expand new to cover left
        new[1] = max(new[1], intervals[i][1])  # expand new to cover right
        i += 1
    result.append(new)                          # append the fully merged new

    # Phase 3: intervals entirely AFTER new
    while i < n:
        result.append(intervals[i])
        i += 1

    return result

# Slow motion on [[1,2],[3,5],[6,7],[8,10],[12,16]] new=[4,8]:
# Phase 1: [1,2] end=2 < 4 → copy. [3,5] end=5 >= 4 → stop.
# Phase 2: [3,5],[6,7],[8,10] all overlap → new=[3,10]
# Phase 3: [12,16] → copy
# result=[[1,2],[3,10],[12,16]]

def test_harness(fn):
    tests = [
        ([[1,3],[6,9]],                         [2,5],  [[1,5],[6,9]]),
        ([[1,2],[3,5],[6,7],[8,10],[12,16]],     [4,8],  [[1,2],[3,10],[12,16]]),
        ([],                                    [5,7],  [[5,7]]),
        ([[1,5]],                               [2,3],  [[1,5]]),
        ([[1,5]],                               [2,7],  [[1,7]]),
        ([[3,5],[12,15]],                       [6,6],  [[3,5],[6,6],[12,15]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        import copy
        got = fn(copy.deepcopy(inputs[0]), copy.deepcopy(inputs[1]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(insert)
print("insert defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Non-Overlapping Intervals — LC 435

---

```
PROBLEM:
  Find minimum number of intervals to remove so none overlap.

TRICK:
  Sort by end time. Greedily KEEP intervals that end earliest.
  If current start < last kept end → overlap → remove current (count++)
  Otherwise keep current (update last_end)

  Why sort by end? The interval ending earliest leaves the most room
  for future intervals — classic interval scheduling maximization.

SLOW MOTION TRACE on [[1,2],[2,3],[3,4],[1,3]]:
  sorted by end: [[1,2],[2,3],[1,3],[3,4]]
  last_end=-inf  removals=0

  [1,2]: 1>=last_end(-inf) → keep, last_end=2
  [2,3]: 2>=2 → keep (touching, not strictly overlap), last_end=3
  [1,3]: 1<3  → OVERLAP → remove, removals=1
  [3,4]: 3>=3 → keep, last_end=4
  answer=1

KEY INSIGHT:
  Sorting by END and keeping earliest-ending intervals is the greedy
  optimal — removing any other interval can only do worse.

TIME:  O(n log n) — sort dominates
SPACE: O(1)       — only last_end and counter
```

In [ ]:
def erase_overlap_intervals(intervals: List[List[int]]) -> int:
    """
    LC 435 — Non-overlapping Intervals
    Approach: sort by end, greedily keep earliest-ending, count removals.
    Args:
        intervals (List[List[int]]): list of [start, end] intervals.
    Returns:
        int: minimum number of intervals to remove.
    Time:  O(n log n) — sort
    Space: O(1)       — two variables
    """
    if not intervals:
        return 0

    intervals.sort(key=lambda x: x[1])   # sort by END — greedy anchor
    last_end = float('-inf')
    removals = 0

    for start, end in intervals:
        if start < last_end:             # overlap with last kept interval
            removals += 1               # remove current (it ends later → worse)
        else:
            last_end = end              # keep this — it ends earliest

    return removals

# Slow motion on [[1,2],[2,3],[3,4],[1,3]] sorted by end:
# sorted: [[1,2],[2,3],[1,3],[3,4]]
# [1,2]: 1>=-inf → keep, last_end=2
# [2,3]: 2>=2   → keep (touching ok), last_end=3
# [1,3]: 1<3    → remove, removals=1
# [3,4]: 3>=3   → keep
# answer=1

def test_harness(fn):
    import copy
    tests = [
        ([[1,2],[2,3],[3,4],[1,3]], 1),
        ([[1,2],[1,2],[1,2]],      2),
        ([[1,2],[2,3]],            0),
        ([[1,4],[2,3],[3,4]],      1),
        ([],                       0),
        ([[1,100],[11,22],[1,11],[2,12]], 2),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(copy.deepcopy(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(erase_overlap_intervals)
print("erase_overlap_intervals defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Minimum Arrows — LC 452

---

```
PROBLEM:
  Fire minimum arrows to burst all balloons [x_start, x_end].
  An arrow at x bursts all balloons where x_start <= x <= x_end.

TRICK:
  Sort by end. Fire arrow at the end of the first balloon.
  Only fire another arrow when the next balloon starts AFTER the current arrow.

  Difference from LC 435: touching counts as HIT (start == arrow → burst).
  So advance arrow only when next.start > arrow (strict >).

SLOW MOTION TRACE on [[10,16],[2,8],[1,6],[7,12]]:
  sorted by end: [[1,6],[2,8],[7,12],[10,16]]
  arrow=6  arrows=1

  [2,8]:  start=2 <=6 → hit by arrow at 6
  [7,12]: start=7 > 6 → new arrow at 12, arrows=2
  [10,16]:start=10<=12 → hit by arrow at 12
  answer=2

KEY INSIGHT:
  Sort by end. Fire arrow at first balloon's end. Every balloon whose
  start <= that position is burst in one shot — no extra arrow needed.

TIME:  O(n log n) — sort dominates
SPACE: O(1)       — arrow position and count
```

In [ ]:
def find_min_arrow_shots(points: List[List[int]]) -> int:
    """
    LC 452 — Minimum Number of Arrows to Burst Balloons
    Approach: sort by end, fire arrow at first end, advance only on miss.
    Args:
        points (List[List[int]]): balloons as [x_start, x_end].
    Returns:
        int: minimum arrows needed.
    Time:  O(n log n) — sort
    Space: O(1)       — arrow variable and count
    """
    if not points:
        return 0

    points.sort(key=lambda x: x[1])   # sort by end coordinate
    arrows = 1
    arrow_pos = points[0][1]          # fire first arrow at end of first balloon

    for start, end in points[1:]:
        if start > arrow_pos:          # balloon starts after current arrow — miss
            arrows += 1
            arrow_pos = end            # fire new arrow at this balloon's end
        # else: start <= arrow_pos → hit by current arrow, no action needed

    return arrows

# Slow motion on [[10,16],[2,8],[1,6],[7,12]] sorted by end:
# sorted: [[1,6],[2,8],[7,12],[10,16]]
# arrow=6, arrows=1
# [2,8]: 2<=6 → hit
# [7,12]: 7>6 → new arrow at 12, arrows=2
# [10,16]: 10<=12 → hit
# answer=2

def test_harness(fn):
    import copy
    tests = [
        ([[10,16],[2,8],[1,6],[7,12]], 2),
        ([[1,2],[3,4],[5,6],[7,8]],    4),
        ([[1,2],[2,3],[3,4],[4,5]],    2),
        ([[1,2]],                      1),
        ([[1,10],[2,3],[4,5],[6,7]],   1),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(copy.deepcopy(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(find_min_arrow_shots)
print("find_min_arrow_shots defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Meeting Rooms — Sweep Line

---

```
PROBLEM:
  Given meeting intervals, find minimum number of rooms needed (LC 253).
  Or: can one person attend all meetings? (LC 252)

TRICK — Sweep Line:
  Create events: +1 at each start, -1 at each end.
  Sort events by time (ends before starts at ties — a meeting ending
  frees a room before the next one occupies it).
  Scan events, track running count, record peak.

SLOW MOTION TRACE on [[0,30],[5,10],[15,20]]:
  events sorted: [(0,+1),(5,+1),(10,-1),(15,+1),(20,-1),(30,-1)]
  time  delta  rooms  peak
    0     +1      1     1
    5     +1      2     2
   10     -1      1     2
   15     +1      2     2
   20     -1      1     2
   30     -1      0     2
  answer=2 rooms

ALTERNATE — TWO SORTED ARRAYS (cleaner for interviews):
  starts = sorted([s for s,e in meetings])
  ends   = sorted([e for s,e in meetings])
  Use two pointers: if starts[i] < ends[j] → need room; else free room.

KEY INSIGHT:
  The peak concurrent count = minimum rooms needed.
  At tie time, sort ends before starts (a freed room can be reused).

TIME:  O(n log n) — sort events or two arrays
SPACE: O(n)       — events list
```

In [ ]:
def min_meeting_rooms(intervals: List[List[int]]) -> int:
    """
    LC 253 — Meeting Rooms II
    Approach: two sorted arrays (starts, ends) with two-pointer sweep.
    Args:
        intervals (List[List[int]]): meeting [start, end] intervals.
    Returns:
        int: minimum rooms required.
    Time:  O(n log n) — two sorts
    Space: O(n)       — two sorted arrays
    """
    if not intervals:
        return 0

    starts = sorted(i[0] for i in intervals)  # all start times
    ends   = sorted(i[1] for i in intervals)  # all end times

    rooms = 0
    peak = 0
    j = 0   # pointer into ends

    for i in range(len(starts)):
        if starts[i] < ends[j]:    # new meeting starts before earliest end
            rooms += 1             # need a new room
        else:
            j += 1                 # a meeting ended — reuse that room
        peak = max(peak, rooms)

    return peak

def can_attend_all(intervals: List[List[int]]) -> bool:
    """
    LC 252 — Meeting Rooms
    Sort by start, check no consecutive overlap.
    Time: O(n log n), Space: O(1)
    """
    intervals.sort(key=lambda x: x[0])
    for i in range(1, len(intervals)):
        if intervals[i][0] < intervals[i-1][1]:   # new starts before last ends
            return False
    return True

# Slow motion on [[0,30],[5,10],[15,20]]:
# starts=[0,5,15] ends=[10,20,30]
# i=0: 0<10 → rooms=1, peak=1
# i=1: 5<10 → rooms=2, peak=2
# i=2: 15>=10 → j=1, rooms=2 (reused), peak=2
# answer=2

def test_harness(fn):
    import copy
    tests = [
        ([[0,30],[5,10],[15,20]], 2),
        ([[7,10],[2,4]],         1),
        ([[1,5],[2,6],[3,7]],    3),
        ([[1,4],[4,5]],          1),
        ([],                     0),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(copy.deepcopy(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(min_meeting_rooms)
print("min_meeting_rooms and can_attend_all defined.")

<a id='10'></a>

## 10. The Intervals Decision Map

```
QUESTION TYPE                        SORT BY   KEY IDEA               LC
────────────────────────────────────────────────────────────────────────────
Merge overlapping intervals          start     extend last merged end  56
Insert into sorted list              —         3-phase scan            57
Min removals → no overlap            end       keep earliest-ending    435
Min arrows to burst balloons         end       fire at first end       452
Min rooms for concurrent meetings    —         two-pointer sweep       253
Can attend all meetings              start     check consecutive gaps  252

SORT CHOICE:
  Sort by START → building the final merged list (LC 56, 252)
  Sort by END   → greedy problems needing most room after kept (LC 435, 452)
  No sort       → already sorted list, just scan (LC 57)
  Two arrays    → meeting rooms sweep (LC 253)
```

<a id='11'></a>

## 11. Interview Cheat Sheet

**1. When to reach for Interval patterns:**

| Signal | What To Do |
|--------|------------|
| Merge overlapping ranges | Sort by start, extend last |
| Insert into sorted list | Three-phase scan |
| Minimize removals / maximize kept | Sort by end, greedy keep |
| Minimum shots/arrows | Sort by end, fire at end |
| Concurrent count (rooms) | Two sorted arrays sweep |

**2. Core operations — memorize these:**

```python
intervals.sort(key=lambda x: x[0])   # sort by start
intervals.sort(key=lambda x: x[1])   # sort by end
overlap = curr[0] <= last[1]         # overlap check (touching inclusive)
last[1] = max(last[1], curr[1])      # extend merged interval
```

**3. Common templates:**

```python
# TEMPLATE 1: MERGE (LC 56)
intervals.sort(key=lambda x: x[0])
merged = [intervals[0]]
for curr in intervals[1:]:
    if curr[0] <= merged[-1][1]: merged[-1][1] = max(merged[-1][1], curr[1])
    else: merged.append(curr)

# TEMPLATE 2: GREEDY KEEP EARLIEST-END (LC 435)
intervals.sort(key=lambda x: x[1])
last_end = float('-inf'); removals = 0
for s, e in intervals:
    if s < last_end: removals += 1
    else: last_end = e

# TEMPLATE 3: MIN ROOMS SWEEP (LC 253)
starts = sorted(i[0] for i in intervals)
ends   = sorted(i[1] for i in intervals)
rooms = peak = j = 0
for i in range(len(starts)):
    if starts[i] < ends[j]: rooms += 1
    else: j += 1
    peak = max(peak, rooms)
```

**4. Gotchas:**

```
❌  Forget to append last merged interval after loop (LC 56)
❌  Wrong boundary in LC 435: use start < last_end (strict), not <=
❌  Wrong boundary in LC 452: use start > arrow (strict), touching = hit
❌  Sort by start when you need sort by end (they solve different problems)
✅  Sort by end for any problem asking to MAXIMIZE what you keep
✅  Sort by start for building/merging the final list
```

<a id='12'></a>

## 12. Summary Map

```
INTERVALS
│
├── Sort by START
│     ├── Merge (extend last)       LC 56
│     └── Attend all (gap check)    LC 252
│
├── Sort by END (greedy: earliest end → most room)
│     ├── Min removals              LC 435
│     └── Min arrows                LC 452
│
├── No sort (already sorted)
│     └── Insert interval           LC 57 — three-phase scan
│
└── Two-Array Sweep
      └── Min concurrent rooms      LC 253

THE OVERLAP FORMULA:
  [a,b] overlaps [c,d]  when  a <= d AND c <= b
  (neither is entirely to the left of the other)

SORT RULE:
  Building final list → sort by START
  Maximizing kept     → sort by END (greedy)
```

---
*End of Intervals Master Guide — Sean Edition*